<a href="https://colab.research.google.com/github/veer-sid0824/Fine-ChatBot/blob/main/FineTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pypdf transformers bitsandbytes accelerate -q


In [ ]:
from google.colab import files
import os

uploaded = files.upload()
file_name = list(uploaded.keys())[0]
print(f"Successfully uploaded: {file_name}")

In [ ]:
!pip install pypdf python-docx -q
!apt-get install -y catdoc -q # Linux utility to read older .doc files

In [ ]:
import os
import subprocess
from pypdf import PdfReader
from docx import Document

def extract_text(file_path):
    # 1. Handle PDF files
    if file_path.endswith('.pdf'):
        reader = PdfReader(file_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() + "\n"
        return text

    # 2. Handle modern Word files (.docx)
    elif file_path.endswith('.docx'):
        doc = Document(file_path)
        text = []
        for paragraph in doc.paragraphs:
            text.append(paragraph.text)
        return "\n".join(text)

    # 3. Handle legacy Word files (.doc)
    elif file_path.endswith('.doc'):
        # Uses the Linux 'catdoc' tool we installed to extract plain text from old binary .doc
        result = subprocess.run(['catdoc', file_path], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        if result.returncode == 0:
            return result.stdout
        else:
            raise ValueError(f"Could not read legacy .doc file. Error: {result.stderr}")

    # 4. Fallback for plain text or configuration files (.txt, .md, etc.)
    else:
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()

# Load your file content (works with .pdf, .docx, .doc, .txt)
document_context = extract_text(file_name)
print(f"Extracted {len(document_context)} characters from the document.")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

# 1. Configure 4-bit quantization to save memory
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# 2. Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 3. Load the model onto the GPU
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
print("Model loaded successfully!")

In [ ]:
def ask_about_document(user_question):
    # Constructing a structured system message to restrict the model's knowledge to your file
    messages = [
        {
            "role": "system",
            "content": f"You are a helpful assistant. Answer the user's question accurately using ONLY the following reference document:\n\n{document_context}"
        },
        {
            "role": "user",
            "content": user_question
        }
    ]

    # Format the prompt into tokens using Qwen's specific template structure
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to("cuda")

    # Generate the text response
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=500,  # Limits response length (increase if you want longer answers)
        temperature=0.1,     # Keeps the model focused and factual based on your text
        do_sample=False
    )

    # Clean up the output so it only displays the answer (skipping the original prompt)
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, generated_ids)]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return response

In [ ]:
import sys

def chat_with_document():
    print("✨ Document Chat Activated! Type your question below.")
    print("👉 Type 'exit' or 'quit' to stop.\n")
    print("-" * 50)

    while True:
        # 1. Take continuous user input
        user_question = input("👤 You: ")

        if user_question.lower() in ['exit', 'quit']:
            print("👋 Chat ended. Happy building!")
            break

        if not user_question.strip():
            continue

        print("\n🤖 Thinking...")

        # 2. Format the payload for Qwen
        messages = [
            {
                "role": "system",
                "content": f"You are a strict, factual assistant. Answer the user's question accurately using ONLY the information provided in this text:\n\n{document_context}"
            },
            {
                "role": "user",
                "content": user_question
            }
        ]

        # 3. Tokenize and generate response on the GPU
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer([text], return_tensors="pt").to("cuda")

        generated_ids = model.generate(
            **inputs,
            max_new_tokens=500,  # Room for detailed answers
            temperature=0.1,     # Keeps it tightly anchored to your text file
            do_sample=False
        )

        # 4. Decode and print output
        generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, generated_ids)]
        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        print(f"\n🤖 Qwen:\n{response}")
        print("-" * 50)

# Start the interactive loop!
chat_with_document()